# Rocket Flight Capstone

The four lessons in this module developed a progression from a physical model to a numerical result that can support an engineering judgment. You derived ordinary differential equations, reconstructed transparent time-stepping methods, located events that fall between stored times, studied convergence, and compared methods at a required accuracy. You also practiced specifying bounded work for an agent and auditing the code it proposed.

This capstone brings those ideas together in a new setting: the vertical flight of a small rocket. The model includes gravity, aerodynamic drag, changing propellant mass, and a known transition from powered flight to coasting flight. You will first work with a transparent fixed-step RK2 calculation and then compare it with an adaptive solver from SciPy. Neither result is authoritative simply because the code runs or comes from a standard library. Your task is to decide what evidence makes the reported flight quantities trustworthy.

(rocket-engineering-brief)=
## Engineering brief

A simulation team needs reference results for an idealized vertical-flight model. These results will be used to check later implementations of the same mathematical model. The quantities of interest are the rocket's maximum speed, apogee, and impact conditions.

The engineering question is:

> Can a fixed-step RK2 calculation and a properly configured adaptive SciPy solver support trustworthy predictions of the rocket's apogee and impact conditions to the required numerical accuracy?

Use the following acceptance targets:

- apogee altitude to within $1\ \mathrm{m}$;
- apogee time to within $0.02\ \mathrm{s}$;
- impact time to within $0.05\ \mathrm{s}$; and
- impact speed to within $0.1\ \mathrm{m/s}$.

These are targets for **numerical error within the stated model**. They do not claim that this simplified model predicts a real rocket to the same accuracy.

Your final verdict will need to identify the evidence supporting an RK2 step size and a SciPy tolerance configuration, explain how burnout and the flight events were handled, and separate numerical uncertainty from limitations of the model. Code generation may be assisted, but you remain responsible for recording the specification of delegated work, auditing the result, and deciding what the evidence supports.

## The vertical-flight problem

We model the rocket as a point mass constrained to move vertically. Altitude $h$ is measured upward from the launch point, and velocity $v=dh/dt$ is positive during ascent and negative during descent. The rocket launches from rest at ground level with $100\ \mathrm{kg}$ of propellant.

| Symbol | Description | Value |
| :--- | :--- | ---: |
| $m_s$ | dry mass of the rocket | $50\ \mathrm{kg}$ |
| $m_{p,0}$ | initial propellant mass | $100\ \mathrm{kg}$ |
| $g$ | gravitational acceleration | $9.81\ \mathrm{m/s^2}$ |
| $\rho$ | air density | $1.091\ \mathrm{kg/m^3}$ |
| $r$ | rocket radius | $0.5\ \mathrm{m}$ |
| $A=\pi r^2$ | reference area | $\pi(0.5\ \mathrm{m})^2$ |
| $C_D$ | drag coefficient | $0.15$ |
| $v_e$ | effective exhaust speed relative to the rocket | $325\ \mathrm{m/s}$ |
| $\mu_0$ | powered-flight propellant burn rate | $20\ \mathrm{kg/s}$ |

The model assumes constant gravity, air density, drag coefficient, reference area, exhaust speed, and burn rate during powered flight. It neglects wind, lateral motion, atmospheric variation, rocket attitude, and any change in aerodynamic properties. The engine switches off when the propellant is exhausted. These assumptions define a useful numerical problem, not a high-fidelity launch model.

## Propellant burn rate

Let $\mu(t)$ denote the **positive** rate at which propellant leaves the rocket. The remaining propellant therefore satisfies $dm_p/dt=-\mu(t)$. With the stated initial mass and constant powered-flight burn rate, burnout occurs at

$$
\label{eq-rocket-burnout-time}
t_b=\frac{m_{p,0}}{\mu_0}=5\ \mathrm{s}.
$$

The prescribed mass-flow history is

$$
\label{eq-rocket-mass-flow}
\mu(t)=
\begin{cases}
\mu_0, & 0\leq t<t_b,\\
0, & t\geq t_b.
\end{cases}
$$

The value changes discontinuously at the known time $t_b$. Later, both numerical approaches will treat burnout as an explicit boundary between two integrations rather than allowing one numerical step to cross it unnoticed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Model parameters.
m_s = 50.0        # dry mass (kg)
m_p0 = 100.0      # initial propellant mass (kg)
g = 9.81          # gravitational acceleration (m/s**2)
rho = 1.091       # air density (kg/m**3)
r = 0.5           # rocket radius (m)
A = np.pi * r**2  # reference area (m**2)
C_D = 0.15        # drag coefficient
v_e = 325.0       # effective exhaust speed (m/s)
mu_0 = 20.0       # powered-flight burn rate (kg/s)
t_burn = m_p0 / mu_0

def mass_flow_rate(t):
    '''Return the positive propellant burn rate at time t.'''
    t = np.asarray(t)
    return np.where((t >= 0.0) & (t < t_burn), mu_0, 0.0)

In [ ]:
t_plot = np.linspace(0.0, 10.0, 201)

fig, ax = plt.subplots(figsize=(6.0, 3.5))
ax.step(t_plot, mass_flow_rate(t_plot), where='post')
ax.axvline(t_burn, color='0.5', linestyle='--', label='burnout')
ax.set_xlabel('Time, $t$ (s)')
ax.set_ylabel('Burn rate, $\mu$ (kg/s)')
ax.set_xlim(0.0, 10.0)
ax.set_ylim(-1.0, 22.0)
ax.grid()
ax.legend()
fig.tight_layout()

Integrating [Equation %s](#eq-rocket-mass-flow) gives the exact remaining propellant mass:

$$
\label{eq-rocket-propellant-history}
m_p(t)=
\begin{cases}
m_{p,0}-\mu_0t, & 0\leq t<t_b,\\
0, & t\geq t_b.
\end{cases}
$$

This exact history will provide a simple but important check of the numerical state. Nonnegative propellant alone is not sufficient: code that applies thrust for too long and then clamps a negative mass to zero still delivers an incorrect impulse to the rocket.

:::{warning .simple .dropdown icon=false open=false} On paper — reproduce the model derivation
Follow the derivation below with upward as the positive direction. Reproduce the mass balance, the momentum balance across a short time interval, and the final three-equation initial-value problem in your paper record. Mark the sign of every force and write the units of each term in the momentum equation.

Keep this derivation beside your computational notebook. During the capstone checkout, you may be asked to explain one step or sign choice.
:::

## Derivation of the equations of motion

### Mass balance

The rocket's instantaneous mass is the sum of its constant dry mass and remaining propellant mass:

$$
\label{eq-rocket-total-mass}
m(t)=m_s+m_p(t).
$$

During a short interval $dt$, a positive mass $dm_e=\mu(t)dt$ leaves the rocket. Consequently,

$$
\label{eq-rocket-propellant-balance}
dm_p=-dm_e=-\mu(t)dt,
\qquad
\frac{dm_p}{dt}=-\mu(t).
$$

Using a separate symbol $\mu$ for the positive outflow rate avoids a common sign ambiguity: $\mu$ is positive while propellant is burning, whereas $dm_p/dt$ is negative.

### Momentum balance and thrust

At the beginning of the interval, the rocket has mass $m$ and upward velocity $v$. At the end, its mass is $m-dm_e$ and its velocity is $v+dv$. The exhaust leaves downward relative to the rocket with speed $v_e$, so to first order its velocity in the stationary reference frame is $v-v_e$.

Let $F_{\mathrm{ext}}$ be the sum of the external forces on the rocket. Equating initial momentum plus the external impulse with final rocket and exhaust momentum gives

$$
\label{eq-rocket-short-time-momentum}
mv+F_{\mathrm{ext}}dt
=(m-dm_e)(v+dv)+dm_e(v-v_e).
$$

Expanding [Equation %s](#eq-rocket-short-time-momentum), canceling equal terms, and neglecting the second-order product $dm_e\,dv$ gives

$$
F_{\mathrm{ext}}dt=m\,dv-dm_e v_e.
$$

Since $dm_e=\mu dt$, the upward-positive equation becomes

$$
\label{eq-rocket-variable-mass-balance}
m\frac{dv}{dt}=F_{\mathrm{ext}}+\mu v_e.
$$

The term $\mu v_e$ is the upward thrust in this model. The exhaust speed is an effective constant: the model does not separately resolve the pressure forces and internal engine flow that determine it.

### Gravity and aerodynamic drag

Gravity acts downward with force $-mg$. Quadratic drag has magnitude $\tfrac12\rho AC_Dv^2$ and always opposes the motion. In one signed velocity component, both directions are represented by

$$
\label{eq-rocket-drag-force}
F_D=-\frac{1}{2}\rho AC_Dv|v|.
$$

When $v>0$, the drag force is negative and opposes ascent. When $v<0$, $v|v|<0$, so the drag force is positive and opposes descent. Replacing $v|v|$ by $v^2$ would silently give the wrong direction during descent.

The total external force is therefore

$$
\label{eq-rocket-external-force}
F_{\mathrm{ext}}=-mg-\frac{1}{2}\rho AC_Dv|v|.
$$

### The initial-value problem

Substituting [Equation %s](#eq-rocket-external-force) into [Equation %s](#eq-rocket-variable-mass-balance), using $m=m_s+m_p$, and adding the altitude and propellant equations gives

$$
\label{eq-rocket-state-equations}
\begin{aligned}
\frac{dh}{dt} &= v,\\
\frac{dv}{dt} &= -g
+\frac{\mu(t)v_e}{m_s+m_p}
-\frac{\rho AC_D}{2(m_s+m_p)}v|v|,\\
\frac{dm_p}{dt} &= -\mu(t).
\end{aligned}
$$

The three initial conditions are

$$
\label{eq-rocket-initial-conditions}
h(0)=0,
\qquad
v(0)=0,
\qquad
m_p(0)=m_{p,0}=100\ \mathrm{kg}.
$$

In vector form, with $u=[h,v,m_p]^T$, the model has the non-autonomous form $u'=f(t,u)$. The explicit appearance of time through $\mu(t)$ is the reason the RK2 step used later must evaluate the second stage at both the midpoint state and the midpoint time.

## Flight regimes and reported events

The solution passes through three physical regimes and three consequential transitions:

| Regime or transition | Numerical condition | What changes or is recorded |
| :--- | :--- | :--- |
| Powered ascent | $0\leq t<t_b$ | $\mu=\mu_0$; mass decreases and thrust acts |
| Burnout | known breakpoint $t=t_b$ | set $m_p=0$, record the state, and continue with $\mu=0$ |
| Coasting ascent | $t>t_b$ and $v>0$ | mass is constant; gravity and drag slow the rocket |
| Apogee | first crossing from $v>0$ to $v\leq0$ | interpolate the time and altitude where $v=0$ |
| Descent | $v<0$ and $h>0$ | gravity acts downward and drag acts upward |
| Impact | first downward crossing from $h>0$ to $h\leq0$ | interpolate the time and velocity where $h=0$ |

Burnout is a **breakpoint known in advance**, while apogee and impact are **events discovered from the computed solution**. Impact must mean the downward ground crossing after the rocket has been aloft; the initial condition $h(0)=0$ is the launch point, not an immediate impact.

## Predictions and checks before computing

A numerical trajectory should be judged against expectations that do not come from the same computation. Three useful checks follow directly from the model.

**Exact propellant history.** Use [Equation %s](#eq-rocket-propellant-history) to calculate the remaining propellant at $t=3.2\ \mathrm{s}$ and confirm that it reaches exactly zero at burnout.

**No-drag burnout-speed bound.** If drag is removed during powered flight, integrating the velocity equation gives

$$
\label{eq-rocket-ideal-burn-velocity}
v_{\mathrm{ideal}}(t)
=v_e\ln\left(\frac{m_s+m_{p,0}}{m_s+m_p(t)}\right)-gt.
$$

During ascent, drag can only reduce the velocity from this ideal value. Evaluate [Equation %s](#eq-rocket-ideal-burn-velocity) at burnout and use it as an upper bound for the powered-flight speed.

**Empty-shell terminal speed.** During descent after burnout, the mass is $m_s$ and a steady downward velocity satisfies $dv/dt=0$. Its magnitude is

$$
\label{eq-rocket-terminal-speed}
v_{\mathrm{terminal}}
=\sqrt{\frac{2m_sg}{\rho AC_D}}.
$$

The rocket begins its descent from rest at apogee, so its impact-speed magnitude should approach this value from below.

Finally, every term in [Equation %s](#eq-rocket-variable-mass-balance) has units of force: $mg$, $\mu v_e$, and $\rho Av^2$ each have units $\mathrm{kg\,m/s^2}$. This dimensional agreement is necessary, although it cannot by itself establish that every sign or coefficient is correct.

:::{warning .simple .dropdown icon=false open=false} On paper — record independent expectations
Before beginning the numerical calculation, your paper record should contain:

- the reproduced mass and momentum derivation;
- the initial-value problem with the state order and sign convention;
- a diagram of the burn, coast, descent, and impact sequence;
- the remaining propellant at $3.2\ \mathrm{s}$;
- the no-drag burnout-speed bound and empty-shell terminal speed; and
- your predicted sign for every force term during ascent and descent.

These expectations become evidence when you later audit the RK2 and SciPy calculations.
:::

---

The narrative and instructional content of this notebook are licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Code cells are licensed under the [BSD 3-Clause License](../../../LICENSES/BSD-3-Clause.txt).